# 03 – Model Training

Demonstrates training the CNN and LSTM models on synthetic data and evaluating performance.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

import config
from src.models_pytorch import ECG_CNN, ECG_LSTM
from src.loss_functions import FocalLoss, compute_class_weights
from src.trainer import Trainer
from src.evaluator import ModelEvaluator
from src.utils import setup_logging, set_seed

setup_logging('INFO')
set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Generate synthetic training data
rng = np.random.RandomState(42)
N_TRAIN, N_VAL, N_TEST = 2000, 400, 400
BEAT_LEN = config.BEAT_WINDOW_SAMPLES

def make_dataset(n, seed=0):
    r = np.random.RandomState(seed)
    X = r.randn(n, BEAT_LEN).astype(np.float32)
    y = r.randint(0, config.NUM_CLASSES, n).astype(np.int64)
    return X, y

X_train, y_train = make_dataset(N_TRAIN, seed=0)
X_val, y_val     = make_dataset(N_VAL, seed=1)
X_test, y_test   = make_dataset(N_TEST, seed=2)
print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)

In [ ]:
# Build DataLoaders
def make_loader(X, y, batch_size=128, shuffle=True):
    X_t = torch.tensor(X).unsqueeze(1)  # (N, 1, 129) for CNN
    y_t = torch.tensor(y)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train)
val_loader   = make_loader(X_val, y_val, shuffle=False)
test_loader  = make_loader(X_test, y_test, shuffle=False)

In [ ]:
# Train CNN for a few epochs
model = ECG_CNN(num_classes=config.NUM_CLASSES)
class_weights = compute_class_weights(y_train, num_classes=config.NUM_CLASSES)
criterion = FocalLoss(alpha=class_weights, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

trainer = Trainer(model, optimizer, criterion, device=device,
                  config={'early_stopping_patience': 5})
history = trainer.train(train_loader, val_loader, epochs=10, save_path=None)
print('Training complete.')

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Val')
ax2.set_title('Accuracy'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
evaluator = ModelEvaluator(class_names=config.AAMI_CLASSES)
y_pred, y_true, y_prob = evaluator.evaluate(trainer.model, test_loader, device)
metrics = evaluator.compute_metrics(y_true, y_pred, y_prob)
print('Macro F1:', round(metrics['macro_f1'], 4))
print('AUC-ROC:', round(metrics.get('auc_roc') or 0, 4))
print(metrics['classification_report'])

In [ ]:
# Confusion matrix
evaluator.plot_confusion_matrix(metrics['confusion_matrix'], class_names=config.AAMI_CLASSES)
plt.show()